# Coupled Models: CICE and BGC

CrocoDash supports two compset-level coupling options beyond standalone ocean:

- **[Section A](#section-a-sea-ice-cice6)** — add CICE6 sea-ice coupling.
- **[Section B](#section-b-biogeochemistry-marbl)** — add MARBL ocean biogeochemistry.

Both follow the same 4-step workflow as the
[Getting Started tutorial](../tutorials/crocodash_tutorial.ipynb).
Only the `compset=` argument and a few `configure_forcings` kwargs change.

## Section A: Sea Ice (CICE6)

Swap the stub `SICE` (stub sea ice) for `CICE` in the compset.
Everything else — grid, topo, vgrid, forcings — is identical to the standalone run.

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot=cesmroot,
    caseroot=caseroot,
    inputdir=inputdir,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    project="NCGD0011",
    override=True,
    machine="derecho",
    compset="GR_JRA",  # GR_JRA = 1850_DATM%JRA_SLND_CICE_MOM6%REGIONAL_SROF_SGLC_SWAV
)

### Optional: Warm start from CICE restart files

After a first run you can use CICE restart files as the ice initial condition
instead of the default (open water). This gives a more realistic sea-ice state
at the start of subsequent runs.

1. Find restart files in your archive: `<caseroot>/archive/rest/<year>/` — look for `*.cice.r.*.nc`.
   If unsure of the archive path, run `./xmlquery DOUT_S_ROOT` in the case directory.
2. Copy the file to your run directory: `cp <restart_file> <run_dir>/`.
3. Open `user_nl_cice` and set `ice_ic = "<restart_filename>"`.

```{note}
History files (`.h` / `.h1`) **cannot** be used as CICE initial conditions — only `.r` restart files.
```

## Section B: Biogeochemistry (MARBL)

MARBL (the Marine Biogeochemistry Library) is bundled with CESM.
Activating it requires:

1. A compset that includes `MOM6%REGIONAL%MARBL-BIO`.
2. A MARBL global initial-condition file.
3. Passing BGC-specific kwargs to `configure_forcings`.

Optionally, combine with river nutrients by also enabling GLOFAS runoff (see the note below).

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot=cesmroot,
    caseroot=caseroot,
    inputdir=inputdir,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    project="NCGD0011",
    override=True,
    machine="derecho",
    compset="CR1850MARBL_JRA",
)

In [ ]:
case.configure_forcings(
    date_range=["2000-01-01 00:00:00", "2000-02-01 00:00:00"],
    data_input_path="",        # path to CESM output data
    product_name="CESM_OUTPUT",
    marbl_ic_filepath="",      # path to MARBL global IC file
    # For river nutrients (requires GLOFAS runoff also enabled):
    # runoff_esmf_mesh_filepath="",
    # global_river_nutrients_filepath="",
)

```{note}
On **Derecho/Casper** the input files are already available:

```{code-block} python
data_input_path = "/glade/campaign/collections/cmip/CMIP6/CESM-HR/FOSI_BGC/HR/g.e22.TL319_t13.G1850ECOIAF_JRA_HR.4p2z.001/ocn/proc/tseries/month_1"
product_name = "CESM_OUTPUT"
marbl_ic_filepath = "/glade/campaign/collections/gdex/data/d651077/cesmdata/inputdata/ocn/mom/tx0.66v1/ecosys_jan_IC_omip_latlon_1x1_180W_c231221.nc"
# River nutrients (optional):
runoff_esmf_mesh_filepath = "/glade/campaign/cesm/cesmdata/cseg/inputdata/ocn/mom/croc/rof/glofas/dis24/GLOFAS_esmf_mesh_v4.nc"
global_river_nutrients_filepath = "/glade/campaign/cesm/cesmdata/cseg/inputdata/ocn/mom/croc/rof/river_nutrients/river_nutrients.GNEWS_GNM.glofas.20250916.64bit.nc"
```
```